In [1]:
import yaml
import pandas as pd
import glob
from pathlib import Path
import json
import os
import sys
import jax.numpy as jnp
sys.path.append("..")

import wandb
api = wandb.Api()

wandb: [wandb.Api()] Loaded credentials for https://api.wandb.ai from WANDB_API_KEY.


In [2]:
root_sweeps = Path("../../wandb-sweep-data-qnrl/")
root_sweeps.mkdir(parents=True, exist_ok=True)

In [3]:
entity_id = "news-vt" # Team name in WandB.
project_id = "qnrl" # Project ID in WandB.

In [4]:
# List of sweep names and their IDs for reference.
sweep_name_id = { # name: id
    # QnRL.
    'qnrl/FrozenLake-v1': '7yoeyqqr',
    'qnrl/CliffWalking-v1': 'nin93kq8',
    'qnrl/Acrobot-v1': '5ehazde3',
    'qnrl/CartPole-v1': 'qojis1fw',

    # C51.
    'c51/FrozenLake-v1': 'jzduwzvg',
    'c51/CliffWalking-v1': '8xyt1eb5',
    'c51/Acrobot-v1': 'n8imxz1v',
    'c51/CartPole-v1': 'arv708xs',
    
    # Breakout-v5.
    'qnrl/Breakout-v5': 'eigl7wk5',
    'c51/Breakout-v5': 'dg4sodfc',
    
    # SpaceInvaders-v5.
    'qnrl/SpaceInvaders-v5': '8adr9d1k',
    'c51/SpaceInvaders-v5': 'wbmr06xp',
}

In [5]:
metadata_path_tree = {
    # Each sweep has a dictionary of runs.
    Path(sweep_dir).stem.split('sweep-')[1]: {
        # Each run has a metadata JSON file.
        Path(run_dir).stem.split('run-')[1]: glob.glob(str(Path(run_dir)/'wandb-metadata.json')) for run_dir in glob.glob(str(Path(sweep_dir)/'*'))
    } for sweep_dir in glob.glob(str(root_sweeps/'*'))
}

In [6]:
summary_path_tree = {
    # Each sweep has a dictionary of runs.
    Path(sweep_dir).stem.split('sweep-')[1]: {
        # Each run has a metadata JSON file.
        Path(run_dir).stem.split('run-')[1]: glob.glob(str(Path(run_dir)/'wandb-summary.json')) for run_dir in glob.glob(str(Path(sweep_dir)/'*'))
    } for sweep_dir in glob.glob(str(root_sweeps/'*'))
}

In [7]:
config_path_tree = {
    # Each sweep has a dictionary of runs.
    Path(sweep_dir).stem.split('sweep-')[1]: {
        # Each run has a config YAML file.
        Path(run_dir).stem.split('run-')[1]: glob.glob(str(Path(run_dir)/'config.yaml')) for run_dir in glob.glob(str(Path(sweep_dir)/'*'))
    } for sweep_dir in glob.glob(str(root_sweeps/'*'))
}

In [8]:
history_path_tree = {
    # Each sweep has a dictionary of runs.
    Path(sweep_dir).stem.split('sweep-')[1]: {
        # Each run has a list of parquet files.
        Path(run_dir).stem.split('run-')[1]: glob.glob(str(Path(run_dir)/'history'/'*.parquet')) for run_dir in glob.glob(str(Path(sweep_dir)/'*'))
    } for sweep_dir in glob.glob(str(root_sweeps/'*'))
}

In [9]:
from collections import defaultdict
def group_runs_by_seed_group_hash(config_run_tree):
    seedgroups_tree = defaultdict(list)
    for run_id, config_paths in config_run_tree.items():
        if config_paths:
            with open(str(config_paths[0]), 'r') as f:
                config = yaml.safe_load(f)
            seed_group_hash = config["seed_group_hash"]["value"]
            seedgroups_tree[seed_group_hash].append(run_id)
    return dict(seedgroups_tree) # Convert to regular dictionary.

seedgroups_tree = {sweep_id: group_runs_by_seed_group_hash(run_tree) for sweep_id, run_tree in config_path_tree.items()}

In [10]:
from functools import reduce
def get_nested_value(d, path):
    return reduce(lambda x,y: x[y], path.split('.'),d)

def build_table_for_env(env_id: str, metrics_to_track: list[str], configs_to_track: list[str]):
    seed_group_metrics = {}
    seed_group_configs = {}
    for seed_group_hash, run_ids in seedgroups_tree[sweep_name_id[env_id]].items():
        seed_group_hash_stub = seed_group_hash.split('-', 1)[0]
        
        metrics = defaultdict(list)
        configs = defaultdict(list)
        for run_id in run_ids:
            summary_path = summary_path_tree[sweep_name_id[env_id]][run_id][0]
            with open(summary_path, 'r') as f:
                summary = json.load(f)
            for metric_key in metrics_to_track:
                metric_stem, metric_agg = metric_key.split('.', 1)
                metrics[metric_key].append(summary[metric_stem][metric_agg])
                
            config_path = config_path_tree[sweep_name_id[env_id]][run_id][0]
            with open(config_path, 'r') as f:
                config = yaml.safe_load(f)
            for config_key in configs_to_track:
                # configs[config_key].append(config['config']['value'][config_key])
                configs[config_key].append(get_nested_value(config['config']['value'], config_key))
        
        seed_group_metrics[seed_group_hash_stub] = {key: jnp.array(vals).mean() for key, vals in metrics.items()} # Aggregate the mean of all metrics for a seed group.
        seed_group_configs[seed_group_hash_stub] = {key: vals[0] for key, vals in configs.items()} # Just take the first one since they are all the same for a seed group.
    return seed_group_metrics, seed_group_configs

## C51

In [11]:
def make_table_c51_classic_control_and_toy_text(env_id: str):
    metrics_to_track = ['eval/episode_reward.mean', 'eval/total_model_parameter_count.mean']
    configs_to_track = ['model.din', 'model.num_actions', 'model.num_atoms', 'z_min', 'z_max', 'model.douts']
    seed_group_metrics, seed_group_configs = build_table_for_env(env_id, metrics_to_track, configs_to_track)
    # Format as dataframe.
    df = pd.concat([pd.DataFrame(seed_group_metrics).T, pd.DataFrame(seed_group_configs).T], axis=1)
    df = df.sort_index(ascending=True) # Sort by seed group hash stub.

    return (
        df.style
        .format({'eval/total_model_parameter_count.mean': lambda x: int(x)})
    )

def make_table_c51_atari(env_id: str):
    metrics_to_track = ['eval/episode_reward.mean', 'eval/total_model_parameter_count.mean']
    configs_to_track = ['model.din', 'model.frame_stack', 'model.w', 'model.h', 'model.num_actions', 'model.num_atoms', 'z_min', 'z_max', 'model.conv_douts', 'model.conv_kernel_size', 'model.conv_strides']
    seed_group_metrics, seed_group_configs = build_table_for_env(env_id, metrics_to_track, configs_to_track)
    # Format as dataframe.
    df = pd.concat([pd.DataFrame(seed_group_metrics).T, pd.DataFrame(seed_group_configs).T], axis=1)
    df = df.sort_index(ascending=True) # Sort by seed group hash stub.

    return (
        df.style
        .format({
            'eval/total_model_parameter_count.mean': lambda x: int(x),
            'model.conv_kernel_size': lambda x: jnp.array(x)[:,0].tolist(),
            'model.conv_strides': lambda x: jnp.array(x)[:,0].tolist(),
        })
    )

def make_table_c51(env_id: str):
    match env_id:
        case 'c51/CartPole-v1':
            styler = make_table_c51_classic_control_and_toy_text(env_id=env_id)
            return styler.format({
                'eval/episode_reward.mean': "{:.3f}",
            })
        case 'c51/CliffWalking-v1':
            styler = make_table_c51_classic_control_and_toy_text(env_id=env_id)
            return styler.format({
                'eval/episode_reward.mean': "{:.1f}",
            })
        case 'c51/FrozenLake-v1':
            styler = make_table_c51_classic_control_and_toy_text(env_id=env_id)
            return styler.format({
                'eval/episode_reward.mean': "{:.4f}",
            })
        case 'c51/Acrobot-v1':
            styler = make_table_c51_classic_control_and_toy_text(env_id=env_id)
            return styler.format({
                'eval/episode_reward.mean': "{:.4f}",
            })
        case 'c51/Breakout-v5':
            styler = make_table_c51_atari(env_id=env_id)
            return styler.format({
                'eval/episode_reward.mean': "{:.2f}",
            })
        case 'c51/SpaceInvaders-v5':
            styler = make_table_c51_atari(env_id=env_id)
            return styler.format({
                'eval/episode_reward.mean': "{:.2f}",
            })
        case _:
            raise ValueError(f"unsupported environment id {env_id}")

In [12]:
make_table_c51(env_id='c51/CartPole-v1')

,eval/episode_reward.mean,eval/total_model_parameter_count.mean,model.din,model.num_actions,model.num_atoms,z_min,z_max,model.douts
4368ff55,224.600,2209.0,4,2,32,-100,100,"[30, 21]"
49e46a87,469.300,16204.0,4,2,32,-100,100,"[120, 84]"
b7aea040,270.150,792.0,4,2,32,-100,100,"[16, 8]"
bdc26ac3,387.225,5614.0,4,2,32,-100,100,"[60, 42]"


In [13]:
make_table_c51(env_id='c51/CliffWalking-v1')

,eval/episode_reward.mean,eval/total_model_parameter_count.mean,model.din,model.num_actions,model.num_atoms,z_min,z_max,model.douts
15173423,-13.0,2072.0,48,4,32,-100,100,"[16, 8]"
8036b0d0,-77.5,590.0,48,4,32,-100,100,"[4, 2]"
8ef48998,-13.0,4937.0,48,4,32,-100,100,"[30, 21]"
b12ce064,-13.5,1151.0,48,4,32,-100,100,"[7, 5]"


In [14]:
make_table_c51(env_id='c51/FrozenLake-v1')

,eval/episode_reward.mean,eval/total_model_parameter_count.mean,model.din,model.num_actions,model.num_atoms,z_min,z_max,model.douts
03b4d6ac,-0.1708,927.0,16,4,32,-2,2,"[7, 5]"
1c44782b,0.1518,1560.0,16,4,32,-2,2,"[16, 8]"
a6bb645e,0.0925,3977.0,16,4,32,-2,2,"[30, 21]"
d9663ea7,-0.3035,462.0,16,4,32,-2,2,"[4, 2]"


In [15]:
make_table_c51(env_id='c51/Acrobot-v1')

,eval/episode_reward.mean,eval/total_model_parameter_count.mean,model.din,model.num_actions,model.num_atoms,z_min,z_max,model.douts
9f7469c2,-500.0000,1112.0,6,3,32,-100,100,"[16, 8]"
c3881283,-320.9000,7110.0,6,3,32,-100,100,"[60, 42]"
d615efcf,-363.3500,2973.0,6,3,32,-100,100,"[30, 21]"
db125df2,-234.0750,19164.0,6,3,32,-100,100,"[120, 84]"


In [16]:
make_table_c51(env_id='c51/SpaceInvaders-v5')

,eval/episode_reward.mean,eval/total_model_parameter_count.mean,model.din,model.frame_stack,model.w,model.h,model.num_actions,model.num_atoms,z_min,z_max,model.conv_douts,model.conv_kernel_size,model.conv_strides
1819f2cb,131.50,680288.0,1,4,84,84,6,32,-10,10,"[32, 64, 64]","[[8, 8], [4, 4], [3, 3]]","[[4, 4], [2, 2], [1, 1]]"
47ffe1c4,79.25,157160.0,1,4,84,84,6,32,-10,10,"[8, 16, 16]","[[8, 8], [4, 4], [3, 3]]","[[4, 4], [2, 2], [1, 1]]"
6feb00c2,71.25,322832.0,1,4,84,84,6,32,-10,10,"[16, 32, 32]","[[8, 8], [4, 4], [3, 3]]","[[4, 4], [2, 2], [1, 1]]"
f758df04,142.25,1.499648e+06,1,4,84,84,6,32,-10,10,"[64, 128, 128]","[[8, 8], [4, 4], [3, 3]]","[[4, 4], [2, 2], [1, 1]]"


In [17]:
make_table_c51(env_id='c51/Breakout-v5')

,eval/episode_reward.mean,eval/total_model_parameter_count.mean,model.din,model.frame_stack,model.w,model.h,model.num_actions,model.num_atoms,z_min,z_max,model.conv_douts,model.conv_kernel_size,model.conv_strides
33e43f68,3.55,222416.0,1,4,84,84,4,32,-10,10,"[16, 32, 32]","[[8, 8], [4, 4], [3, 3]]","[[4, 4], [2, 2], [1, 1]]"
99dbcbf7,1.80,106920.0,1,4,84,84,4,32,-10,10,"[8, 16, 16]","[[8, 8], [4, 4], [3, 3]]","[[4, 4], [2, 2], [1, 1]]"
9c1e6421,5.20,1.098176e+06,1,4,84,84,4,32,-10,10,"[64, 128, 128]","[[8, 8], [4, 4], [3, 3]]","[[4, 4], [2, 2], [1, 1]]"
beb2a303,3.55,479520.0,1,4,84,84,4,32,-10,10,"[32, 64, 64]","[[8, 8], [4, 4], [3, 3]]","[[4, 4], [2, 2], [1, 1]]"


## QnRL

In [18]:
def make_table_qnrl_classic_control_and_toy_text(env_id: str):
    metrics_to_track = ['eval/episode_reward.mean', 'eval/total_model_parameter_count.mean']
    configs_to_track = ['model.din', 'model.n_actions', 'model.n_returns', 'model.z_min', 'model.z_max', 'model.n_reuploads', 'model.m_moment', 'model.n_layers', 'model.pauli_enc', 'model.pauli_var', 'model.imprimitive', 'model.input_multiplex_activation']
    seed_group_metrics, seed_group_configs = build_table_for_env(env_id, metrics_to_track, configs_to_track)
    # Format as dataframe.
    df = pd.concat([pd.DataFrame(seed_group_metrics).T, pd.DataFrame(seed_group_configs).T], axis=1)
    df = df.sort_index(ascending=True) # Sort by seed group hash stub.

    return (
        df.style
        .format({'eval/total_model_parameter_count.mean': lambda x: int(x)})
    )

def make_table_qnrl_atari(env_id: str):
    metrics_to_track = ['eval/episode_reward.mean', 'eval/total_model_parameter_count.mean']
    configs_to_track = ['model.din', 'model.frame_stack', 'model.w', 'model.h', 'model.n_actions', 'model.n_returns', 'model.z_min', 'model.z_max', 'model.n_reuploads', 'model.m_moment', 'model.n_layers', 'model.pauli_enc', 'model.pauli_var', 'model.imprimitive', 'model.input_multiplex_activation', 'model.conv_douts', 'model.conv_kernel_size', 'model.conv_strides']
    seed_group_metrics, seed_group_configs = build_table_for_env(env_id, metrics_to_track, configs_to_track)
    # Format as dataframe.
    df = pd.concat([pd.DataFrame(seed_group_metrics).T, pd.DataFrame(seed_group_configs).T], axis=1)
    df = df.sort_index(ascending=True) # Sort by seed group hash stub.

    return (
        df.style
        .format({
            'eval/total_model_parameter_count.mean': lambda x: int(x),
            'model.conv_kernel_size': lambda x: jnp.array(x)[:,0].tolist(),
            'model.conv_strides': lambda x: jnp.array(x)[:,0].tolist(),
        })
    )

def make_table_qnrl(env_id: str):
    match env_id:
        case 'qnrl/CartPole-v1':
            styler = make_table_qnrl_classic_control_and_toy_text(env_id=env_id)
            return styler.format({
                'eval/episode_reward.mean': "{:.3f}",
            })
        case 'qnrl/CliffWalking-v1':
            styler = make_table_qnrl_classic_control_and_toy_text(env_id=env_id)
            return styler.format({
                'eval/episode_reward.mean': "{:.1f}",
            })
        case 'qnrl/FrozenLake-v1':
            styler = make_table_qnrl_classic_control_and_toy_text(env_id=env_id)
            return styler.format({
                'eval/episode_reward.mean': "{:.4f}",
            })
        case 'qnrl/Acrobot-v1':
            styler = make_table_qnrl_classic_control_and_toy_text(env_id=env_id)
            return styler.format({
                'eval/episode_reward.mean': "{:.4f}",
            })
        case 'qnrl/Breakout-v5':
            styler = make_table_qnrl_atari(env_id=env_id)
            return styler.format({
                'eval/episode_reward.mean': "{:.2f}",
            })
        case 'qnrl/SpaceInvaders-v5':
            styler = make_table_qnrl_atari(env_id=env_id)
            return styler.format({
                'eval/episode_reward.mean': "{:.2f}",
            })
        case _:
            raise ValueError(f"unsupported environment id {env_id}")

In [19]:
make_table_qnrl(env_id='qnrl/CartPole-v1')

,eval/episode_reward.mean,eval/total_model_parameter_count.mean,model.din,model.n_actions,model.n_returns,model.z_min,model.z_max,model.n_reuploads,model.m_moment,model.n_layers,model.pauli_enc,model.pauli_var,model.imprimitive,model.input_multiplex_activation
0a1d6db2,336.150,330.0,4,2,32,-100,100,2,2,7,['RX'],"['RZ', 'RY', 'RZ']",offset-CZ,tanh
14a05a3d,241.625,330.0,4,2,32,-100,100,2,1,7,['RX'],"['RZ', 'RY', 'RZ']",circ-CZ,tanh
171a91d1,336.150,330.0,4,2,32,-100,100,1,2,7,['RX'],"['RZ', 'RY', 'RZ']",offset-CZ,tanh
1f3b03a6,168.150,250.0,4,2,32,-100,100,2,2,5,['RX'],"['RZ', 'RY', 'RZ']",offset-CZ,tanh
1f8ffc6f,168.150,250.0,4,2,32,-100,100,1,2,5,['RX'],"['RZ', 'RY', 'RZ']",offset-CZ,tanh
31994c7a,241.625,330.0,4,2,32,-100,100,1,1,7,['RX'],"['RZ', 'RY', 'RZ']",circ-CZ,tanh
3a00cecd,256.475,250.0,4,2,32,-100,100,1,1,5,['RX'],"['RZ', 'RY', 'RZ']",circ-CZ,tanh
405bd6e3,43.025,170.0,4,2,32,-100,100,2,2,3,['RX'],"['RZ', 'RY', 'RZ']",offset-CZ,tanh
45913de1,452.300,330.0,4,2,32,-100,100,1,1,7,['RX'],"['RZ', 'RY', 'RZ']",offset-CZ,tanh
45b9d666,256.475,250.0,4,2,32,-100,100,2,1,5,['RX'],"['RZ', 'RY', 'RZ']",circ-CZ,tanh


In [20]:
make_table_qnrl(env_id='qnrl/CliffWalking-v1')

,eval/episode_reward.mean,eval/total_model_parameter_count.mean,model.din,model.n_actions,model.n_returns,model.z_min,model.z_max,model.n_reuploads,model.m_moment,model.n_layers,model.pauli_enc,model.pauli_var,model.imprimitive,model.input_multiplex_activation
08d163cb,-34.5,1380.0,48,4,32,-100,100,1,1,5,['RX'],"['RZ', 'RY', 'RZ']",offset-CZ,tanh
0b5c3b76,-56.0,1380.0,48,4,32,-100,100,1,2,5,['RX'],"['RZ', 'RY', 'RZ']",circ-CZ,tanh
12c045e6,-13.0,1220.0,48,4,32,-100,100,1,1,3,['RX'],"['RZ', 'RY', 'RZ']",circ-CZ,tanh
1729a977,-56.0,1540.0,48,4,32,-100,100,1,2,7,['RX'],"['RZ', 'RY', 'RZ']",circ-CZ,tanh
1fc6943d,-35.0,1540.0,48,4,32,-100,100,4,1,7,['RX'],"['RZ', 'RY', 'RZ']",offset-CZ,tanh
2d18e0c2,-13.0,1220.0,48,4,32,-100,100,4,1,3,['RX'],"['RZ', 'RY', 'RZ']",circ-CZ,tanh
2f5083d1,-77.5,1220.0,48,4,32,-100,100,1,2,3,['RX'],"['RZ', 'RY', 'RZ']",circ-CZ,tanh
3992cb09,-13.0,1380.0,48,4,32,-100,100,1,1,5,['RX'],"['RZ', 'RY', 'RZ']",circ-CZ,tanh
3c16ef96,-13.0,1540.0,48,4,32,-100,100,1,2,7,['RX'],"['RZ', 'RY', 'RZ']",offset-CZ,tanh
773dd844,-34.5,1540.0,48,4,32,-100,100,1,1,7,['RX'],"['RZ', 'RY', 'RZ']",circ-CZ,tanh


In [21]:
make_table_qnrl(env_id='qnrl/FrozenLake-v1')

,eval/episode_reward.mean,eval/total_model_parameter_count.mean,model.din,model.n_actions,model.n_returns,model.z_min,model.z_max,model.n_reuploads,model.m_moment,model.n_layers,model.pauli_enc,model.pauli_var,model.imprimitive,model.input_multiplex_activation
09fc2cfb,0.0990,580.0,16,4,32,-2,2,4,2,3,['RX'],"['RZ', 'RY', 'RZ']",circ-CZ,tanh
0beeeeea,0.1623,900.0,16,4,32,-2,2,1,2,7,['RX'],"['RZ', 'RY', 'RZ']",offset-CZ,tanh
1a6e23d3,0.0480,580.0,16,4,32,-2,2,1,1,3,['RX'],"['RZ', 'RY', 'RZ']",offset-CZ,tanh
2a30337c,0.0990,580.0,16,4,32,-2,2,1,2,3,['RX'],"['RZ', 'RY', 'RZ']",circ-CZ,tanh
2cae8b90,0.1815,900.0,16,4,32,-2,2,4,1,7,['RX'],"['RZ', 'RY', 'RZ']",circ-CZ,tanh
35d94c95,0.1937,740.0,16,4,32,-2,2,4,2,5,['RX'],"['RZ', 'RY', 'RZ']",offset-CZ,tanh
3cee6e98,0.2500,740.0,16,4,32,-2,2,1,2,5,['RX'],"['RZ', 'RY', 'RZ']",circ-CZ,tanh
4270289a,0.2120,740.0,16,4,32,-2,2,4,1,5,['RX'],"['RZ', 'RY', 'RZ']",offset-CZ,tanh
485c0bf8,0.1622,900.0,16,4,32,-2,2,4,2,7,['RX'],"['RZ', 'RY', 'RZ']",offset-CZ,tanh
5064f085,0.2120,740.0,16,4,32,-2,2,1,1,5,['RX'],"['RZ', 'RY', 'RZ']",offset-CZ,tanh


In [22]:
make_table_qnrl(env_id='qnrl/Acrobot-v1')

,eval/episode_reward.mean,eval/total_model_parameter_count.mean,model.din,model.n_actions,model.n_returns,model.z_min,model.z_max,model.n_reuploads,model.m_moment,model.n_layers,model.pauli_enc,model.pauli_var,model.imprimitive,model.input_multiplex_activation
01f1b2a8,-204.2750,405.0,6,3,32,-100,100,3,1,5,['RX'],"['RZ', 'RY', 'RZ']",circ-CZ,tanh
03b176ff,-500.0000,405.0,6,3,32,-100,100,3,2,5,['RX'],"['RZ', 'RY', 'RZ']",offset-CZ,tanh
043b80e2,-500.0000,525.0,6,3,32,-100,100,3,1,7,['RX'],"['RZ', 'RY', 'RZ']",circ-CZ,tanh
0ff9fb8e,-500.0000,285.0,6,3,32,-100,100,3,2,3,['RX'],"['RZ', 'RY', 'RZ']",offset-CZ,tanh
10d163eb,-415.7750,405.0,6,3,32,-100,100,3,1,5,['RX'],"['RZ', 'RY', 'RZ']",offset-CZ,tanh
1bacc532,-196.4250,405.0,6,3,32,-100,100,1,2,5,['RX'],"['RZ', 'RY', 'RZ']",circ-CZ,tanh
24689683,-419.9000,525.0,6,3,32,-100,100,1,2,7,['RX'],"['RZ', 'RY', 'RZ']",offset-CZ,tanh
25417497,-415.7750,405.0,6,3,32,-100,100,1,1,5,['RX'],"['RZ', 'RY', 'RZ']",offset-CZ,tanh
283bce61,-428.6750,285.0,6,3,32,-100,100,3,2,3,['RX'],"['RZ', 'RY', 'RZ']",circ-CZ,tanh
43229ab1,-500.0000,525.0,6,3,32,-100,100,1,1,7,['RX'],"['RZ', 'RY', 'RZ']",circ-CZ,tanh


In [23]:
make_table_qnrl(env_id='qnrl/SpaceInvaders-v5')

,eval/episode_reward.mean,eval/total_model_parameter_count.mean,model.din,model.frame_stack,model.w,model.h,model.n_actions,model.n_returns,model.z_min,model.z_max,model.n_reuploads,model.m_moment,model.n_layers,model.pauli_enc,model.pauli_var,model.imprimitive,model.input_multiplex_activation,model.conv_douts,model.conv_kernel_size,model.conv_strides
28cf3201,91.50,69494.0,1,4,84,84,6,32,-10,10,1,1,7,['RX'],"['RZ', 'RY', 'RZ']",offset-CZ,tanh,"[16, 32, 32]","[[8, 8], [4, 4], [3, 3]]","[[4, 4], [2, 2], [1, 1]]"
2a976698,88.75,69494.0,1,4,84,84,6,32,-10,10,1,1,7,['RX'],"['RZ', 'RY', 'RZ']",circ-CZ,tanh,"[16, 32, 32]","[[8, 8], [4, 4], [3, 3]]","[[4, 4], [2, 2], [1, 1]]"
3f1e580c,84.75,172694.0,1,4,84,84,6,32,-10,10,1,1,5,['RX'],"['RZ', 'RY', 'RZ']",offset-CZ,tanh,"[32, 64, 64]","[[8, 8], [4, 4], [3, 3]]","[[4, 4], [2, 2], [1, 1]]"
4031053e,49.50,172934.0,1,4,84,84,6,32,-10,10,1,1,7,['RX'],"['RZ', 'RY', 'RZ']",offset-CZ,tanh,"[32, 64, 64]","[[8, 8], [4, 4], [3, 3]]","[[4, 4], [2, 2], [1, 1]]"
4801d71d,99.25,69254.0,1,4,84,84,6,32,-10,10,1,1,5,['RX'],"['RZ', 'RY', 'RZ']",circ-CZ,tanh,"[16, 32, 32]","[[8, 8], [4, 4], [3, 3]]","[[4, 4], [2, 2], [1, 1]]"
56217398,83.25,484262.0,1,4,84,84,6,32,-10,10,1,1,7,['RX'],"['RZ', 'RY', 'RZ']",circ-CZ,tanh,"[64, 128, 128]","[[8, 8], [4, 4], [3, 3]]","[[4, 4], [2, 2], [1, 1]]"
5a7a5c51,97.25,484022.0,1,4,84,84,6,32,-10,10,1,1,5,['RX'],"['RZ', 'RY', 'RZ']",circ-CZ,tanh,"[64, 128, 128]","[[8, 8], [4, 4], [3, 3]]","[[4, 4], [2, 2], [1, 1]]"
70aae234,73.50,30830.0,1,4,84,84,6,32,-10,10,1,1,7,['RX'],"['RZ', 'RY', 'RZ']",offset-CZ,tanh,"[8, 16, 16]","[[8, 8], [4, 4], [3, 3]]","[[4, 4], [2, 2], [1, 1]]"
7fbdd5b1,53.50,172934.0,1,4,84,84,6,32,-10,10,1,1,7,['RX'],"['RZ', 'RY', 'RZ']",circ-CZ,tanh,"[32, 64, 64]","[[8, 8], [4, 4], [3, 3]]","[[4, 4], [2, 2], [1, 1]]"
8066810a,154.50,484262.0,1,4,84,84,6,32,-10,10,1,1,7,['RX'],"['RZ', 'RY', 'RZ']",offset-CZ,tanh,"[64, 128, 128]","[[8, 8], [4, 4], [3, 3]]","[[4, 4], [2, 2], [1, 1]]"


In [24]:
make_table_qnrl(env_id='qnrl/Breakout-v5')

,eval/episode_reward.mean,eval/total_model_parameter_count.mean,model.din,model.frame_stack,model.w,model.h,model.n_actions,model.n_returns,model.z_min,model.z_max,model.n_reuploads,model.m_moment,model.n_layers,model.pauli_enc,model.pauli_var,model.imprimitive,model.input_multiplex_activation,model.conv_douts,model.conv_kernel_size,model.conv_strides
05815f17,1.10,22700.0,1,4,84,84,4,32,-10,10,1,1,7,['RX'],"['RZ', 'RY', 'RZ']",circ-CZ,tanh,"[8, 16, 16]","[[8, 8], [4, 4], [3, 3]]","[[4, 4], [2, 2], [1, 1]]"
069ad0e5,1.95,53204.0,1,4,84,84,4,32,-10,10,1,1,3,['RX'],"['RZ', 'RY', 'RZ']",circ-CZ,tanh,"[16, 32, 32]","[[8, 8], [4, 4], [3, 3]]","[[4, 4], [2, 2], [1, 1]]"
1a8fc558,2.75,141284.0,1,4,84,84,4,32,-10,10,1,1,7,['RX'],"['RZ', 'RY', 'RZ']",circ-CZ,tanh,"[32, 64, 64]","[[8, 8], [4, 4], [3, 3]]","[[4, 4], [2, 2], [1, 1]]"
2153e7e5,1.30,22380.0,1,4,84,84,4,32,-10,10,1,1,3,['RX'],"['RZ', 'RY', 'RZ']",offset-CZ,tanh,"[8, 16, 16]","[[8, 8], [4, 4], [3, 3]]","[[4, 4], [2, 2], [1, 1]]"
21892ef5,0.40,22380.0,1,4,84,84,4,32,-10,10,1,1,3,['RX'],"['RZ', 'RY', 'RZ']",circ-CZ,tanh,"[8, 16, 16]","[[8, 8], [4, 4], [3, 3]]","[[4, 4], [2, 2], [1, 1]]"
48b698e2,3.15,421092.0,1,4,84,84,4,32,-10,10,1,1,5,['RX'],"['RZ', 'RY', 'RZ']",circ-CZ,tanh,"[64, 128, 128]","[[8, 8], [4, 4], [3, 3]]","[[4, 4], [2, 2], [1, 1]]"
49fee01c,1.85,53204.0,1,4,84,84,4,32,-10,10,1,1,3,['RX'],"['RZ', 'RY', 'RZ']",offset-CZ,tanh,"[16, 32, 32]","[[8, 8], [4, 4], [3, 3]]","[[4, 4], [2, 2], [1, 1]]"
4d2222f7,4.90,421092.0,1,4,84,84,4,32,-10,10,1,1,5,['RX'],"['RZ', 'RY', 'RZ']",offset-CZ,tanh,"[64, 128, 128]","[[8, 8], [4, 4], [3, 3]]","[[4, 4], [2, 2], [1, 1]]"
4dbdc680,0.85,53364.0,1,4,84,84,4,32,-10,10,1,1,5,['RX'],"['RZ', 'RY', 'RZ']",circ-CZ,tanh,"[16, 32, 32]","[[8, 8], [4, 4], [3, 3]]","[[4, 4], [2, 2], [1, 1]]"
55bfe7f3,2.40,53364.0,1,4,84,84,4,32,-10,10,1,1,5,['RX'],"['RZ', 'RY', 'RZ']",offset-CZ,tanh,"[16, 32, 32]","[[8, 8], [4, 4], [3, 3]]","[[4, 4], [2, 2], [1, 1]]"
